# 01 · 전처리 & PDF 변환 (온도 시리즈)

**워크플로**
1. 폴더 불러오기 — 서브폴더 이름이 온도(예: `300K`), 각 폴더 안에 `.dm4`
2. 전처리 전/후 비교 — hot pixel 제거 · median · beam center
3. PDF 변환 과정 — I(q) → φ(q) → G(r)
4. 온도별 결과를 각각 `.npz`로 저장

재사용 함수는 모두 `fourdstem` 패키지에 들어 있습니다.
실제 `.dm4` 데이터가 없으면 `USE_SYNTHETIC=True`로 두면 데모용 합성 데이터로 전 셀이 실행됩니다.

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import fourdstem as fds

# ── 사용자 설정 ──────────────────────────────────────────────
DATA_ROOT = "data/SiOx_Tseries"     # 온도로 이름 붙은 .dm4 폴더 (예: 0025K.dm4 …) 또는 서브폴더
OUT_DIR   = "rdf_npz"               # 온도별 npz 저장 폴더
USE_SYNTHETIC = not os.path.isdir(DATA_ROOT)   # 실데이터 없으면 자동 데모
N_JOBS = 4                          # 병렬 코어 수. 큰 4D 파일은 로딩 메모리 때문에 4~8 권장
LAZY   = True                       # 큰 4D(수 GB) 파일: memmap으로 청크 평균 → RAM 최소. 작은 파일이면 False

# ⚑ q 단위 힌트: 일부 DM(dm4)은 검출기 역격자 단위를 '1/nm'로 잘못 저장합니다.
#   그 경우 "1/A"로 강제해서 잘못된 ÷10 을 막습니다. (단위가 정상이면 None)
Q_UNIT_HINT = "1/A"

# reduction 파라미터 — 이 패키지는 q = 1/d 컨벤션.
#   전자 PDF는 보통 max q(1/d) ~ 1.5 1/Å 근처 → 구간을 그에 맞춤 (0.8~12은 X!).
#   시리즈 전체에서 LOCK (온도 간 비교 가능하게).
CFG = fds.RDFConfig(composition={"Si": 1, "O": 2},
                    q_int_min=0.15, q_int_max=1.50, r_min=1.10,
                    r_max=8.0, dr=0.02, damping="lorch")
os.makedirs(OUT_DIR, exist_ok=True)
print("USE_SYNTHETIC =", USE_SYNTHETIC, "| N_JOBS =", N_JOBS, "| cores =", os.cpu_count())

## 1) 폴더 불러오기

`fds.Series.from_directory(root)` 는 **레이아웃을 자동 감지**해서 온도 시리즈를 만듭니다:

- **평평한 파일** — 폴더 안에 온도로 이름 붙은 `.dm4` (예: `0025K.dm4 … 1100K.dm4`) →
  파일명에서 온도 파싱
- **서브폴더** — 온도별 서브폴더(예: `300K/scan.dm4`) → 폴더명에서 온도 파싱

`preprocess=` 훅으로 로드 즉시 hot/dead pixel 제거를 적용하고, `n_jobs`로 여러 파일을 **병렬 로드**,
`progress=True`로 **진행바**를 봅니다. `0025K` 처럼 앞에 0이 붙어도, `25K`/`300K`/`1100K` 모두 정확히 파싱됩니다.

**메모리 (큰 4D 파일)**: 파일 하나가 수 GB인 4D 큐브(예: 150×150×256×256 ≈ 5.8 GB)면, `lazy=True`로
**memmap 청크 평균**을 써서 파일을 통째로 RAM에 올리지 않습니다. 병렬 로딩은 동시에 여러 파일을 여니
`n_jobs`는 `4~8`처럼 보수적으로(코어 32개라도) 두는 게 안전합니다. RDF 계산 자체(작은 256×256)는
이후 `series.map`에서 코어를 많이 써도 됩니다.

In [ ]:
def make_ring(shape=(256, 256), r1=70.0, r2=120.0, hot=True, seed=0):
    '''데모용 비정질 링 패턴 (+ beam stop, hot pixel).'''
    rng = np.random.default_rng(seed)
    H, W = shape
    yy, xx = np.mgrid[0:H, 0:W]
    cx, cy = W/2 + 3, H/2 - 2
    r = np.hypot(xx - cx, yy - cy)
    img = 4.0*np.exp(-r**2/(2*5**2))
    img += 1.0*np.exp(-(r-r1)**2/(2*14**2)) + 0.4*np.exp(-(r-r2)**2/(2*10**2))
    img += 0.03*rng.standard_normal(shape)
    img[:, W//2-1:W//2+1] = 0.0                     # beam stopper rod
    if hot:
        for _ in range(8):
            img[rng.integers(H), rng.integers(W)] += 3e3
    return np.clip(img, 0, None)

if USE_SYNTHETIC:
    temps = [300, 400, 500, 600, 700, 800, 900]
    frames = [fds.Frame(pattern=make_ring(seed=T, r1=70+0.0*T), coord=float(T),
                        label=f"{T}K", q_per_px=0.0125) for T in temps]
    series = fds.Series(frames)
else:
    # 평평한 파일(0025K.dm4 …)이든 서브폴더(300K/…)든 자동 감지.
    # q_unit_hint로 단위 강제, n_jobs로 병렬 로드, progress로 진행바.
    series = fds.Series.from_directory(DATA_ROOT, q_unit_hint=Q_UNIT_HINT,
                                       preprocess=fds.clean_pattern,
                                       n_jobs=N_JOBS, progress=True, lazy=LAZY)

print(f"{len(series)} frames:", series.labels())
print("temperatures:", series.coordinates())

# q 캘리브레이션 점검 — 데이터 q 범위가 변환 구간(CFG)과 겹치는지 확인
f0 = series[0]
q_per_px = f0.q_per_px or 1.0
q_max = q_per_px * min(f0.pattern.shape) / 2
print(f"\nq_per_px = {q_per_px:.4g} 1/Å/px,  data max q ≈ {q_max:.3g} 1/Å")
print(f"FT window = [{CFG.q_int_min}, {CFG.q_int_max}] 1/Å")
if q_max < CFG.q_int_max * 0.5:
    print("⚠️  data max q 가 변환 구간보다 훨씬 작습니다 — Q_UNIT_HINT='1/A' 필요하거나 "
          "CFG 구간을 낮추세요 (q=1/d 컨벤션).")

## 2) 전처리 전/후 — hot pixel · median · beam center

한 프레임을 골라 **원본 → hot pixel 제거 → beam stop 마스크 + Friedel 중심**을 나란히 봅니다.
(`from_folders`에 `preprocess`를 넣으면 이 정리는 이미 적용되어 있으니, 여기선 비교용으로 원본을 다시 만듭니다.)

In [ ]:
frame = series[0]
raw = make_ring(seed=int(frame.coord)) if USE_SYNTHETIC else frame.pattern
q_per_px = frame.q_per_px or 1.0

# 전처리
cleaned, hot_mask = fds.remove_hot_pixels(raw, threshold=8, return_mask=True)
cleaned = fds.remove_dead_pixels(cleaned)
stopper = fds.beam_stopper_mask(cleaned)
(cx, cy), fried = fds.find_center(cleaned, stopper)
print(f"hot pixels removed: {hot_mask.sum()},  center=({cx:.1f},{cy:.1f}),  Friedel={fried:.2f}")

fig, ax = plt.subplots(1, 3, figsize=(14, 4.4))
fds.show_pattern(raw, ax=ax[0], title="raw")
fds.show_pattern(cleaned, ax=ax[1], title="hot/dead removed")
fds.show_pattern(cleaned, center=(cx, cy), mask=stopper, ax=ax[2],
                 title="beam stop mask + center")
fig.tight_layout(); plt.show()

## 3) PDF 변환 과정 — I(q) → φ(q) → G(r)

`fds.pattern_to_rdf` 한 번으로 방위각 적분 → 환원 → 사인 변환까지 수행하고,
중간 산출물(`Iq`, `phi`, `Gr`)을 그대로 들여다볼 수 있습니다.

In [ ]:
res = fds.pattern_to_rdf(cleaned, q_per_px, CFG, center=(cx, cy))

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
fds.plot_profile(res.q, res.Iq, ax=ax[0], ylabel="I(q)", title="(1) azimuthal I(q)")
fds.plot_profile(res.q_reduced, res.phi, ax=ax[1], ylabel="φ(q)",
                 title="(2) reduced phi(q)")
ax[1].axhline(0, color="0.7", lw=0.8)
fds.plot_rdf(res, ax=ax[2], title="(3) G(r)")
fig.suptitle(f"PDF pipeline  (N={res.N:.3g})", y=1.03)
fig.tight_layout(); plt.show()

## 4) 온도별 처리 & npz 저장 (병렬 + 진행바)

시리즈 전체를 돌면서 **중심·스케일 N만 프레임별**로 다시 잡고(나머지는 LOCK), 각 온도의 결과를
`{온도}K_rdf.npz`로 저장합니다. `series.map(func, n_jobs=N_JOBS, progress=True)`가 **여러 코어로 병렬
처리**하고 **진행바**를 보여줍니다(32코어면 `N_JOBS=-1`로 32개 모두 사용). 계산은 병렬로, **저장은
메인에서 순서대로** 하여 순서를 보장합니다. `fds.save_rdf`는 `q, Iq, φ, r, Gr`와 스칼라를 함께
저장하므로 02 노트북에서 전체 시리즈를 그대로 다시 불러올 수 있습니다.

> **합성 모드 주의**: 실데이터 파이프라인의 G(r) 첫 피크 위치는 q-범위에 의해 고정되어 온도에 따라
> 잘 움직이지 않습니다. 데모에서 02 노트북(NMF·가우시안 피팅)을 의미 있게 보여주기 위해,
> 합성 모드에서는 **물리적으로 그럴듯한 G(r)를 직접 합성**해 저장합니다(첫 배위 1.60→1.66 Å 이동 +
> 두 번째 성분 증가). 실데이터에서는 `USE_SYNTHETIC=False`로 실제 파이프라인이 저장합니다.

In [ ]:
def synth_Gr(r, T, Tmin=300, Tmax=900):
    '''두 end-member(A: 저온, B: 고온)를 온도에 따라 섞은 데모 G(r).'''
    t = (T - Tmin) / (Tmax - Tmin)
    w = t*t*(3 - 2*t)                       # smoothstep 0→1
    def shell(c, s, a):  return a*np.exp(-(r-c)**2/(2*s**2))
    A = shell(1.60, 0.09, 1.0) + shell(2.55, 0.14, 0.5)
    B = shell(1.66, 0.07, 1.2) + shell(2.62, 0.11, 0.6) + shell(3.10, 0.10, 0.3)
    g = (1-w)*A + w*B
    g = g - 2.2*r*np.exp(-r)               # 저-r 음의 기울기(현실감)
    return g

# 프레임별 RDF 계산(부작용 없는 순수 함수) → series.map으로 병렬 실행
def process_frame(f):
    if USE_SYNTHETIC:
        r = np.arange(0.0, CFG.r_max, CFG.dr)
        return dict(coord=f.coord, label=f.label, r=r, Gr=synth_Gr(r, f.coord),
                    synthetic=True)
    rr = fds.pattern_to_rdf(f.pattern, f.q_per_px or 1.0, CFG)
    return dict(coord=f.coord, label=f.label, result=rr, synthetic=False)

# 진행바와 함께 병렬 계산 (합성 모드는 가벼워 순차)
results = series.map(process_frame, n_jobs=(1 if USE_SYNTHETIC else N_JOBS),
                     progress=True, desc="RDF")

# 저장·요약은 메인 프로세스에서 순서대로 (결정적 순서 보장)
summary = []
for out in results:
    path = os.path.join(OUT_DIR, f"{int(out['coord'])}K_rdf.npz")
    if out["synthetic"]:
        fds.save_result_npz(path, r=out["r"], Gr=out["Gr"], q=res.q, Iq=res.Iq,
                            temperature=out["coord"], source=out["label"])
        r1, _ = fds.first_peak_position(out["r"], out["Gr"], 1.5, 1.7)
    else:
        rr = out["result"]
        fds.save_rdf(path, rr, temperature=out["coord"], source=out["label"])
        r1, _ = fds.first_peak_position(rr.r, rr.Gr, 1.3, 2.2)
    summary.append((out["coord"], r1, path))
    print(f"  T={out['coord']:>5.0f}K  r1={r1:.3f} Å  →  {os.path.basename(path)}")

print(f"\n저장 완료: {len(summary)} 개 npz → {OUT_DIR}/  (N_JOBS={N_JOBS})")

## 5) 온도별 결과 한눈에 (모든 온도 겹쳐 보기)

저장한 npz를 모두 불러와 **모든 온도의 I(q)와 G(r)를 무지개 워터폴**로 겹쳐 봅니다.
(온도에 따른 변화가 한 그림에 보입니다. 개별 곡선을 크게 보려면 02 노트북에서 NMF까지 이어집니다.)

In [ ]:
import glob
files = sorted(glob.glob(os.path.join(OUT_DIR, "*_rdf.npz")))
recs = []
for p in files:
    d = fds.load_result_npz(p)
    recs.append((float(d["temperature"]), np.asarray(d["q"]), np.asarray(d["Iq"]),
                 np.asarray(d["r"]), np.asarray(d["Gr"])))
recs.sort(key=lambda t: t[0])
temps = np.array([t for t, *_ in recs])

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
fds.plot_series_waterfall([(q, Iq) for _, q, Iq, _, _ in recs], temps, ax=ax[0],
                          cmap="rainbow", xlabel="q (1/Å)",
                          labels=[f"{int(T)}K" for T in temps])
ax[0].set_title("I(q) — 온도별 (rainbow waterfall)")
fds.plot_series_waterfall([(r, Gr) for _, _, _, r, Gr in recs], temps, ax=ax[1],
                          cmap="rainbow", xlabel="r (Å)")
ax[1].set_title("G(r) — 온도별 (rainbow waterfall)")
plt.tight_layout(); plt.show()

다음: **02 노트북**에서 이 `npz`들을 불러와 NMF 분해와 첫 피크 이동을 분석합니다.